In [1]:
%pip install -q -U gradio pandas langchain langchain-cohere

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 44.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


In [2]:

import pandas as pd
import gradio as gr

from getpass import getpass
from langchain_cohere import ChatCohere
from langchain.tools import tool
from langchain.agents import create_agent

# STEP 1: Enter Cohere API key
api_key = getpass("Enter your Cohere API key: ")

# STEP 2: Initialize the LLM
llm = ChatCohere(
    model="command-a-03-2025",
    cohere_api_key=api_key,
    temperature=0
)

# STEP 3: Shared dataset
df = None


# STEP 4: Create analysis tools

@tool
def dataset_overview() -> str:
    """Show dataset shape, columns, data types, missing values
    and descriptive statistics."""
    if df is None:
        return "Please upload a CSV file first."

    return (
        f"Shape: {df.shape}\n\n"
        f"Columns and types:\n{df.dtypes.to_string()}\n\n"
        f"Missing values:\n{df.isnull().sum().to_string()}\n\n"
        f"Statistics:\n{df.describe(include='all').to_string()}"
    )


@tool
def product_analysis(product: str) -> str:
    """Analyze a treadmill product such as TM195, TM498 or TM798.
    Returns customer count and average age, income, usage,
    fitness and miles for that product."""
    if df is None:
        return "Please upload a CSV file first."

    if "Product" not in df.columns:
        return "The dataset has no Product column."

    matches = df[
        df["Product"].astype(str).str.upper()
        == product.strip().upper()
    ]

    if matches.empty:
        return f"No records found for product {product}."

    numeric_columns = [
        c for c in
        ["Age", "Education", "Usage", "Fitness", "Income", "Miles"]
        if c in matches.columns
    ]

    return (
        f"Product: {product}\n"
        f"Customer count: {len(matches)}\n\n"
        f"Average values:\n"
        f"{matches[numeric_columns].mean().round(2).to_string()}"
    )


@tool
def compare_products() -> str:
    """Compare treadmill models by customer count and average
    age, income, usage, fitness and miles."""
    if df is None:
        return "Please upload a CSV file first."

    if "Product" not in df.columns:
        return "The dataset has no Product column."

    numeric_columns = [
        c for c in
        ["Age", "Education", "Usage", "Fitness", "Income", "Miles"]
        if c in df.columns
    ]

    result = df.groupby("Product").agg(
        Customer_Count=("Product", "size"),
        **{
            f"Average_{c}": (c, "mean")
            for c in numeric_columns
        }
    )

    return result.round(2).to_string()


@tool
def category_analysis(column: str) -> str:
    """Count the frequency of values in a categorical column.
    For example, Gender, Product or MaritalStatus."""
    if df is None:
        return "Please upload a CSV file first."

    if column not in df.columns:
        return (
            f"Column not found: {column}. "
            f"Available columns: {list(df.columns)}"
        )

    return df[column].value_counts(dropna=False).to_string()


@tool
def correlation_analysis() -> str:
    """Calculate correlations between numerical dataset columns."""
    if df is None:
        return "Please upload a CSV file first."

    numeric_df = df.select_dtypes(include="number")

    if numeric_df.shape[1] < 2:
        return "At least two numeric columns are required."

    return numeric_df.corr().round(2).to_string()


@tool
def filter_customers(
    min_age: int = 0,
    max_age: int = 100,
    min_income: int = 0,
    product: str = "ALL"
) -> str:
    """Filter customers by age, minimum income and product.
    Example: min_age=25, max_age=40, min_income=50000,
    product='TM798'. Use ALL to include all products."""
    if df is None:
        return "Please upload a CSV file first."

    required = {"Age", "Income", "Product"}

    if not required.issubset(df.columns):
        return "Dataset must contain Age, Income and Product columns."

    result = df[
        df["Age"].between(min_age, max_age)
        & (df["Income"] >= min_income)
    ]

    if product.strip().upper() != "ALL":
        result = result[
            result["Product"].astype(str).str.upper()
            == product.strip().upper()
        ]

    if result.empty:
        return "No customers match these conditions."

    return (
        f"Matching customers: {len(result)}\n\n"
        f"{result.head(20).to_string(index=False)}"
    )


# STEP 5: Register tools with the agent
tools = [
    dataset_overview,
    product_analysis,
    compare_products,
    category_analysis,
    correlation_analysis,
    filter_customers
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a data analysis agent for CardioGood Fitness.

    Use the available tools to calculate answers from the uploaded
    dataset. Do not invent numbers or claim you calculated something
    without using a tool.

    Explain results in simple business language.
    Distinguish observed patterns from causal conclusions.
    If the dataset cannot answer a question, say so.
    """
)


# STEP 6: Upload and load the CSV
def load_dataset(file_path):
    global df

    if file_path is None:
        df = None
        return "Please upload CardioGoodFitness.csv."

    try:
        new_df = pd.read_csv(file_path)

        if new_df.empty:
            df = None
            return "The CSV contains no data."

        # Accept the new dataset only after successful loading
        df = new_df

        return (
            "Dataset loaded successfully!\n"
            f"Rows: {df.shape[0]}\n"
            f"Columns: {df.shape[1]}\n"
            f"Column names: {list(df.columns)}\n\n"
            "You can now ask analysis questions."
        )

    except Exception as e:
        df = None
        return f"Could not load CSV: {e}"


# STEP 7: Run the agent
def analyze_data(question):
    if df is None:
        return "Upload the CSV and click Load Dataset first."

    if not question or not question.strip():
        return "Enter a question."

    try:
        result = agent.invoke({
            "messages": [
                {"role": "user", "content": question}
            ]
        })

        # Get the final response
        for message in reversed(result["messages"]):
            if message.type == "ai" and message.content:
                return message.content

        return "The agent did not return an answer."

    except Exception as e:
        return f"Agent error: {e}"


# STEP 8: Create Gradio interface
with gr.Blocks(title="CardioGood Fitness Data Agent") as demo:

    gr.Markdown("# CardioGood Fitness Data Analysis Agent")
    gr.Markdown(
        "Upload your CSV and ask questions in natural language."
    )

    csv_file = gr.File(
        label="Upload CardioGoodFitness.csv",
        file_types=[".csv"],
        type="filepath"
    )

    load_button = gr.Button(
        "Load Dataset",
        variant="primary"
    )

    status = gr.Textbox(
        label="Dataset Status",
        lines=4,
        interactive=False
    )

    load_button.click(
        fn=load_dataset,
        inputs=csv_file,
        outputs=status
    )

    question = gr.Textbox(
        label="Ask a data analysis question",
        placeholder="Compare income across treadmill models"
    )

    analyze_button = gr.Button("Analyze Data")

    answer = gr.Textbox(
        label="Agent Analysis",
        lines=15,
        interactive=False
    )

    analyze_button.click(
        fn=analyze_data,
        inputs=question,
        outputs=answer
    )

    question.submit(
        fn=analyze_data,
        inputs=question,
        outputs=answer
    )

demo.launch(share=True)


Enter your Cohere API key: ··········
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8459224467e2e11175.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:

import pandas as pd
import requests
import gradio as gr

from getpass import getpass
from langchain_cohere import ChatCohere
from langchain.tools import tool
from langchain.agents import create_agent

# --------------------------------------------------
# STEP 1: Initialize Cohere
# --------------------------------------------------

api_key = getpass("Enter your Cohere API key: ")

llm = ChatCohere(
    model="command-a-03-2025",
    cohere_api_key=api_key,
    temperature=0
)

# Shared dataset for this single-user demonstration
df = None


# --------------------------------------------------
# STEP 2: Dataset analysis tool
# --------------------------------------------------

@tool
def analyze_dataset(question: str) -> str:
    """
    Analyze the uploaded CardioGood Fitness dataset.
    Use for customer counts, averages, product comparisons,
    distributions, correlations and customer segments.
    The question describes what analysis is needed.
    """
    if df is None:
        return "No dataset is loaded."

    question_lower = question.lower()

    # Product comparison
    if any(word in question_lower for word in
           ["compare", "comparison", "highest", "lowest", "across"]):

        numeric_cols = [
            c for c in
            ["Age", "Education", "Usage", "Fitness", "Income", "Miles"]
            if c in df.columns
        ]

        if "Product" in df.columns:
            result = df.groupby("Product").agg(
                Customer_Count=("Product", "size"),
                **{
                    "Average_" + c: (c, "mean")
                    for c in numeric_cols
                }
            )

            return result.round(2).to_string()

    # Correlation analysis
    if "correlation" in question_lower or "correlated" in question_lower:
        numeric_df = df.select_dtypes(include="number")

        return numeric_df.corr().round(2).to_string()

    # Dataset overview
    if any(word in question_lower for word in
           ["overview", "missing", "summary", "statistics"]):
        return (
            f"Shape: {df.shape}\n\n"
            f"Missing values:\n{df.isnull().sum().to_string()}\n\n"
            f"Summary statistics:\n"
            f"{df.describe(include='all').to_string()}"
        )

    # Frequency analysis
    if "gender" in question_lower and "Gender" in df.columns:
        return df["Gender"].value_counts().to_string()

    if "product" in question_lower and "Product" in df.columns:
        return df["Product"].value_counts().to_string()

    # General numerical summary
    numeric_df = df.select_dtypes(include="number")

    return (
        f"Available columns: {list(df.columns)}\n\n"
        f"Numerical summary:\n{numeric_df.describe().round(2).to_string()}"
    )


# --------------------------------------------------
# STEP 3: External API tool
# --------------------------------------------------

@tool
def get_current_weather(city: str) -> str:
    """
    Call an external weather API to retrieve current weather.
    Use when a question asks about current weather or fitness
    conditions in a city.
    """
    try:
        # API 1: Find the city's coordinates
        geo_url = "https://geocoding-api.open-meteo.com/v1/search"

        geo_response = requests.get(
            geo_url,
            params={
                "name": city,
                "count": 1,
                "language": "en",
                "format": "json"
            },
            timeout=15
        )

        geo_response.raise_for_status()
        geo_data = geo_response.json()

        locations = geo_data.get("results", [])

        if not locations:
            return f"Could not find the city: {city}"

        location = locations[0]
        latitude = location["latitude"]
        longitude = location["longitude"]
        city_name = location["name"]
        country = location.get("country", "")

        # API 2: Retrieve current weather
        weather_url = "https://api.open-meteo.com/v1/forecast"

        weather_response = requests.get(
            weather_url,
            params={
                "latitude": latitude,
                "longitude": longitude,
                "current": (
                    "temperature_2m,relative_humidity_2m,"
                    "apparent_temperature,precipitation,"
                    "weather_code,wind_speed_10m"
                ),
                "timezone": "auto"
            },
            timeout=15
        )

        weather_response.raise_for_status()
        weather = weather_response.json()["current"]

        return (
            f"City: {city_name}, {country}\n"
            f"Observation time: {weather.get('time')}\n"
            f"Temperature: {weather.get('temperature_2m')} "
            f"{weather.get('units', {}).get('temperature_2m', '°C')}\n"
            f"Feels like: {weather.get('apparent_temperature')} °C\n"
            f"Humidity: {weather.get('relative_humidity_2m')}%\n"
            f"Precipitation: {weather.get('precipitation')} mm\n"
            f"Wind speed: {weather.get('wind_speed_10m')} km/h\n"
            f"Weather code: {weather.get('weather_code')}"
        )

    except Exception as e:
        return f"Weather API error: {str(e)}"


# --------------------------------------------------
# STEP 4: Create the agent and register its tools
# --------------------------------------------------

tools = [
    analyze_dataset,
    get_current_weather
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a CardioGood Fitness data analysis agent.

    You have two tools:
    1. analyze_dataset: analyzes the uploaded customer dataset.
    2. get_current_weather: retrieves current weather using an API.

    Decide which tool or tools are needed for each question.
    You may call multiple tools if the question requires both.

    Explain the results clearly.
    Never invent API results or numerical calculations.
    Distinguish observed data from assumptions.
    Weather alone is not sufficient to determine whether
    exercise is safe; avoid making medical claims.
    """
)


# --------------------------------------------------
# STEP 5: Load the uploaded CSV
# --------------------------------------------------

def load_dataset(file_path):
    global df

    if file_path is None:
        df = None
        return "Please upload a CSV file."

    try:
        new_df = pd.read_csv(file_path)

        if new_df.empty:
            df = None
            return "The uploaded CSV has no rows."

        df = new_df

        return (
            f"Dataset loaded successfully.\n"
            f"Rows: {df.shape[0]}\n"
            f"Columns: {df.shape[1]}\n"
            f"Column names: {list(df.columns)}"
        )

    except Exception as e:
        df = None
        return f"CSV loading error: {e}"


# --------------------------------------------------
# STEP 6: Ask the agent a question
# --------------------------------------------------

def ask_agent(question):
    if not question or not question.strip():
        return "Please enter a question."

    if df is None:
        return "Please upload the CSV and click Load Dataset first."

    try:
        result = agent.invoke({
            "messages": [
                {"role": "user", "content": question}
            ]
        })

        # Retrieve the final AI response
        for message in reversed(result["messages"]):
            if message.type == "ai" and message.content:
                return message.content

        return "The agent did not return an answer."

    except Exception as e:
        return f"Agent error: {str(e)}"


# --------------------------------------------------
# STEP 7: Build the Gradio interface
# --------------------------------------------------

with gr.Blocks(title="Data Analysis Agent with APIs") as demo:

    gr.Markdown("# CardioGood Fitness Data Analysis Agent")
    gr.Markdown(
        "Upload a dataset and ask questions involving "
        "customer analytics or live weather API data."
    )

    csv_file = gr.File(
        label="Upload CardioGoodFitness.csv",
        file_types=[".csv"],
        type="filepath"
    )

    load_button = gr.Button("Load Dataset", variant="primary")
    status = gr.Textbox(label="Dataset Status", lines=3)

    load_button.click(
        fn=load_dataset,
        inputs=csv_file,
        outputs=status
    )

    question = gr.Textbox(
        label="Ask a question",
        placeholder=(
            "Compare product usage and get current weather "
            "for Chennai."
        )
    )

    ask_button = gr.Button("Ask Agent")
    answer = gr.Textbox(
        label="Agent Response",
        lines=12
    )

    ask_button.click(
        fn=ask_agent,
        inputs=question,
        outputs=answer
    )

    question.submit(
        fn=ask_agent,
        inputs=question,
        outputs=answer
    )

demo.launch(share=True)


Enter your Cohere API key: ··········
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d56ea4e1999628545e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
